**Masked language modeling** seemed like a promising idea, but since the model only predicts one token, not one word, expanding words that aren't just one token is not possible. One example is `conf.`. Here the expansion could, e.g., be `confirmare`, but for `ClassCat/roberta-base-latin-v2` this would be split into `confirm` and `are` and for `Cicciokr/XLM-Roberta-Base-Latin-Uncased` it would split into `confirmar` and `e`, which means that masking just the `.` (`conf<mask>`) is not possible and neither is masking the whole thing. Even with `latincy/latin-bert`, where `confirmare` is actually part of the vocabulary, this just definitely wouldn't work for all words (e.g. `beneficiatus`, `consanguinitas` or `archipresbiteratus` are not part of the vocabulary), making the predictions significantly/heavily biased toward shorter words. Using multiple masks to match the tokenization also wouldn't work, since different expansions might be tokenized into a different number of tokens (and another problem is that RoBERTa definitely and it seems also BERT fill multiple masks independently).

In [ ]:
import polars as pl
from openai import OpenAI
from ratelimit import limits, sleep_and_retry
from datetime import date
import json

# API configuration
base_url = "https://chat-ai.academiccloud.de/v1"
model = "openai-gpt-oss-120b" # decent
#model = "apertus-70b-instruct-2509" # terrible
#model = "deepseek-r1-distill-llama-70b" # bad
#model = "gemma-4-31b-it" # decent? - doesn't expand what it shouldn't - does expand things that it shouldn't/hallucinates (expanded `A` without a `.`) when there is nothing to be expanded, but it makes sense to catch that case anyway
    # also did smart selection of expansion (used the expansion `(gratia ) expectativa` as just `expectativa`) - this could be both good and bad (doing smart adjustsments is good, but sometimes this could lead to errors)
    # Tru[d]perti' became 'Trudperti
#model = "glm-4.7" # decent?
#model = "mistral-large-3-675b-instruct-2512" # terrible? (didn't respect #5: Scope)
#model = "qwen3.5-397b-a17b" # terrible? - HUGE amount of reasoning and bad result (only so far tested on one example)
# mschonhardt/latin-normalizer terrible (for expanding) - leaves out stuff and halucinates (might be useful for actually normalising later though)
# andbue/byt5-base-latin-normalize useless for expanding
# dantedgp/latin-english-MT completely useless
# hathibelagal/llama-3.2-latin seemed promising, but my prompt (Please expand all abbreviations in the following text) for some reason leads to no output being generated.
api_key = "Key"
ONE_MINUTE = 60
STEP_SIZE = 10
MAX_BATCH_ATTEMPTS = 10
MAX_ROW_ATTEMPTS = 5
DEBUG = True

client = OpenAI(api_key=api_key, base_url=base_url)

SYSTEM_PROMPT = """**Role:** You are a historian specializing in medieval church history with expert knowledge of the Latin abbreviations used in the papal registers.

**Task:** You will receive a Latin text in which some abbreviations are marked as `[[id|abbreviation]]`, together with a list of expansion candidates for each id. For every id, select the candidate that best fits the grammatical and semantic context of the surrounding text.

**Instructions:**

1. **Choices:** Choose exactly one candidate per id, from the given candidates only.

2. **Base forms:** Return the chosen candidate exactly as it is written in the candidate list. Do not inflect, alter, or extend it — the grammatical form is adjusted in a later processing step.

3. **Occurrences:** The same abbreviation can require different expansions at different places in the text; judge each occurrence in its own context. If you are unsure, prefer the expansion most commonly associated with that abbreviation in medieval Latin church documents.

4. **Output format:** Return only a single JSON object mapping every id to the chosen candidate, e.g. `{"1": "confirmatio", "2": "dominus"}`. Do not include explanations, commentary, or any other text.
"""

# earlier full-text-rewrite versions of the prompt, kept for reference:
# the model returned the whole expanded text, which sometimes corrupted words
# it should not have touched (e.g. `Halberstad.` → `Halhalstad.`) and required
# a fragile token diff (check_expansion_errors.py) for validation
full_rewrite_version = """**Role:** You are a historian specializing in medieval church history with expert knowledge of Latin abbreviations. You will receive a Latin text containing abbreviations, along with a list of expansion candidates for each abbreviation. Your task is to select the most appropriate candidate for each abbreviation and return the fully expanded text.

**Instructions:**

1. **Primary task:** Expand each abbreviation using only the provided candidates. If multiple candidates are listed for an abbreviation, choose the one that best fits the grammatical and semantic context of the surrounding text. If you are unsure, prefer the expansion most commonly associated with that abbreviation in medieval Latin church documents.

2. **Grammatical forms:** The candidates represent base (dictionary) forms of words. You may inflect the chosen word into the grammatical form required by the context (e.g., adjusting case, number, gender, or tense). For example, if the candidate is `dominus` but the context requires a genitive, expand the abbreviation as `domini`.

3. **Consistency:** If the same abbreviation appears multiple times in the text, expand it consistently unless the context clearly calls for different expansions.

4. **Format:** Return only the expanded text, preserving the original formatting, punctuation, and word order. Do not include explanations, commentary, the original abbreviated text, or the candidate list in your response.

5. **Scope:** Do not expand or alter any part of the text that is not listed as an abbreviation, even if you suspect it may be abbreviated.
"""

first_version = """**Role:** You are a historian specializing in medieval church history with good knowledge of latin abbreviations. You will receive a latin text that contains some abbreviations and a list of expansion candidates for most or all abbreviations. Your task is to choose a candidate for each listed abbreviation and return the expanded text.

**Instructions:**
1. **Primary task:** Expand abbreviations using only the provided candidates. If there are multiple candidates for an abbreviation, choose the one that best fits the context of the text. If you are unsure, choose the most common expansion for that abbreviation in medieval church history.
2. **Format:** Return only the expanded text without any additional explanations or formatting. Do not include the original text or the list of candidates in your response.
3. **Contextual understanding:** Use your knowledge of medieval church history to make informed decisions about which expansions are most likely correct in the context of the text.
"""

In [2]:
@sleep_and_retry
@limits(calls=15, period=ONE_MINUTE)
def call_chat_ai(user_prompt: str):
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {"role": "user", "content": user_prompt},
        ],
        model=model,
        temperature=0,
    )
    return chat_completion.model_dump()

In [3]:
ablass_ids = pl.read_csv("data/ablaesse.csv").unique(subset="RG Nr.").with_columns(pl.col("RG Nr.").str.split_exact("/", 1).struct.rename_fields(["volume", "nr_RG"]).struct.unnest()).with_columns(
    pl.col("volume").cast(pl.Int64),
    pl.col("nr_RG").cast(pl.Int64),
).select("volume", "nr_RG")

In [4]:
rg = pl.read_csv("data/RG_header_sublemma_all.csv").select(["volume","nr_RG","nr_suffix","header_no_tags","regest_no_tags","id_RG_all"]).join(ablass_ids,how="inner", on=["volume", "nr_RG"])
once_expanded = pl.read_csv("data/expanded.csv").join(ablass_ids,how="inner", on=["volume", "nr_RG"])
#pl.read_csv("data/complex.csv").with_columns(pl.col("Auflösung").str.replace_all(';','').str.replace_all(r'\(.+\)','').str.strip_chars()).write_csv("data/complex_copy.csv")
simple = pl.read_csv("data/simple.csv")
complex = pl.read_csv("data/complex_copy.csv")
# my_additions.csv is not needed here, because all of the expansions in that csv are unique for all volumes
# TODO for now I'll drop the information in RG1-RG9, but for some entries there is actually valuable information in there

In [5]:
def vita_df_to_text(vita:pl.DataFrame):
    header = vita.get_column("header_no_tags").drop_nulls().item() # implicit assertion that there is just one header
    regests = vita.sort(by=["volume", "nr_RG", "nr_suffix"]).get_column("regest_no_tags").drop_nulls().implode().item().to_list()
    return f"{header}\n{'\n'.join(regests)}"

In [6]:
def text_to_vita_df(text: str, volume: int, nr: int):
    pieces = text.split('\n')

    header = [None for piece in pieces]
    regests = pieces

    header[0] = pieces[0]
    regests[0] = None

    vita = pl.DataFrame(
        {"volume" : [volume for piece in pieces],
        "nr_RG": [nr for piece in pieces],
        "nr_suffix": range(len(pieces)),
        "header_no_tags": header,
        "regest_no_tags": regests,
        "id_RG_all": [f"1{volume:02d}{nr:05d}-{i}" for i in range(len(pieces))]})
    return vita


In [16]:
#volume, nr = 5, 885
volume, nr = 5, 1954
vita = rg.filter((pl.col("volume") == volume) & (pl.col("nr_RG") == nr))
print(vita_df_to_text(vita))

Fredericus dux Saxonie et marchio Misnen., lantgravius Thuringie et Sigismundus dux Saxonie et marchio modernus Misnen. ac capit. colleg. eccl. s. Georgii in castro Aldenburg Nuemburg. dioc., cuius mense capitul. 200 m. arg. p. olim quond. Wilhelmus marchio Misnen. predium nonnullorum mansorum in Aldenburg ac allodia c. pertinentiis et par. eccll. in Elsterberg, Kale, Ffroberg, Koryn, Gotznitz, Borgwerbin, Lugkaw, Hinna, Czeegerucke et Luben Magunt., Halberstad. et Nuemburg. dioc. insimul 60 m. arg. p. , quarum ius patron. ad d. marchionem pertinebat, donavit, incorp. a Johanne XXIII. et Nicolao [de Lubich] ep. Merseburg. approbata
de conf. d. incorp. 26. apr. 1432 S 276 153rs.
et Sigismundus, duces Saxonie: de lic. elig. confess., alt. port., ante diem, de locis interd. etiam pro ux. et fam. 26. apr. 32 S 276 153vs, de rem. plen. etiam pro ux. 26. apr. 1432 S 276 154r.
dux Saxonie et elector S.R.I., qui nuper oratores ad papam destinavit: breve quo mittit copiam revocationis transl. c

**Multiple-choice reformat:** instead of having the model rewrite the whole text (which sometimes corrupted words it shouldn't touch, e.g. `Halberstad.` → `Halhalstad.`, silently expanded abbreviations without glossary entries, and required a fragile token diff for validation), the abbreviations with candidates are marked in the text as `[[id|abbreviation]]` and the model only returns a JSON object mapping each id to the chosen candidate. The substitution happens programmatically (`multiple_choice.py`), so the rest of the text cannot change and every choice is validated by a simple membership test against the candidate list. Chosen candidates are inserted in their base form; the inflection is left to the normalisation step.

In [ ]:
from helper_functions import find_abbreviations
from multiple_choice import find_candidate_occurrences, build_user_prompt, parse_choices, apply_choices

def determine_candidates(vita: pl.DataFrame, simple: pl.DataFrame, complex: pl.DataFrame):
    volume = vita.get_column("volume").unique().item() # implicit assertion that there is just one vita and consequently one volume in the dataframe
    simple = simple.filter(pl.col(f"^RG{volume}$").is_not_null())
    simple = simple.filter(pl.col("Abkürzung").is_duplicated())

    abbreviations = find_abbreviations(vita)
    multiword_abbreviations_with_spaces = abbreviations.filter(abbreviations.str.find(r"\.\w").is_not_null()).str.replace_all(r"\.(\w)", r". $1")
    abbreviations = pl.concat((abbreviations, multiword_abbreviations_with_spaces))

    simple_candidates = pl.DataFrame(abbreviations).join(simple, on="Abkürzung", how="inner").select("Abkürzung", "Auflösung").group_by("Abkürzung").agg(pl.col("Auflösung"))
    complex_candidates = pl.DataFrame(abbreviations).join(complex, on="Abkürzung", how="inner").select("Abkürzung", "Auflösung").group_by("Abkürzung").agg(pl.col("Auflösung"))
    
    candidates = pl.concat((simple_candidates, complex_candidates)).sort(by="Abkürzung")
    return {row["Abkürzung"]: row["Auflösung"] for row in candidates.iter_rows(named=True)}

def expand_single(original_text: str, candidates: dict[str, list[str]]):

    occurrences = find_candidate_occurrences(original_text, candidates)
    if not occurrences:
        print("nothing to be expanded")
        return None

    user_prompt = build_user_prompt(original_text, occurrences, candidates)
    #print(user_prompt)

    choices, errors, dump = None, [], None
    for attempt in range(MAX_ROW_ATTEMPTS):
        dump = call_chat_ai(user_prompt)
        content = dump["choices"][0]["message"]["content"]
        choices, errors = parse_choices(content, occurrences, candidates)
        if choices is not None:
            break
    if choices is None:
        # no parseable response after all attempts -> leave everything unexpanded
        choices = {}

    return {
        "text": apply_choices(original_text, occurrences, choices),
        "choices": [
            {"id": occ.id, "abbreviation": occ.matched, "choice": choices.get(occ.id)}
            for occ in occurrences
        ],
        "errors": errors,
        "user_prompt": user_prompt,
        "response": dump,
    }

In [40]:
volume = 5
#nr = 370
nr = 885

vita_df = once_expanded.filter((pl.col("volume") == volume) & (pl.col("nr_RG") == nr))
once_expanded_text = vita_df_to_text(vita_df)
candidates = determine_candidates(vita_df, simple, complex)

print(once_expanded_text)
print('-'*100 + '\n')
for key in candidates:
    print(f"{key}:")
    for value in candidates[key]:
        print(f"    {value}")

Brunswic Brunswicen. Halberstad. et Hildesem. diocc.
abbas et monasterium s. Egidii B. ordo sancti Benedicti Halberstad. diocesis: de conserv. 30. iun. 1435 S 307 228vs.
par. ecclesia sancti Andree B. Hildesem. diocesis Ludolpho Quirre archidiaconus in Stockem in eccl. Hildesem. et rector d. parochialis ecclesia supplic. : de indulg. 30. iun. 1435 S 309 203r.
decanus, capitulum et singuli canonici collegiata ecclesia sancti Blasii B. unius de notabilioribus collegiata eccl. Saxonie Ottone, Wilhelmo et Hinrico Brunswicen. et Luneborgen. ducibus, patron. etiam supplic. : de incorporare parochialis ecclesia in Woden Weden Hildesem. diocesis 4 marca argenti fabrice d. ecclesia sancti Blasii 2 marca argenti 13. october 1438 S 350 168vs.
proconsules, consules et universitas op. B. Hildesem. et Halberstad. diocc.: de conserv. privilegium de non evocando eis a Sigismundo R.I. conc. et a Martino V. conf. 26. iun. 1436 S 323 235vs, exec.: abbas monasterium sancti Petri et Pauli in Regaliluttere 

In [ ]:
result = expand_single(once_expanded_text, candidates)
if result:
    twice_expanded_text = result["text"]
    print(twice_expanded_text)

In [ ]:
for choice in result["choices"]:
    print(f"[[{choice['id']}]] {choice['abbreviation']} → {choice['choice']}")

print()
for error in result["errors"]:
    print(error["message"])

In [43]:
twice_expanded_df = text_to_vita_df(twice_expanded_text, volume, nr)
twice_expanded_df

volume,nr_RG,nr_suffix,header_no_tags,regest_no_tags,id_RG_all
i64,i64,i64,str,str,str
5,885,0,"""Brunswic Brunswicen. Halhalsta…",null,"""10500885-0"""
5,885,1,null,"""abbas et monasterium sancti Eg…","""10500885-1"""
5,885,2,null,"""parochialis ecclesia sancti An…","""10500885-2"""
5,885,3,null,"""decanus, capitulum et singuli …","""10500885-3"""
5,885,4,null,"""proconsules, consules et unive…","""10500885-4"""


# expanding a batch

In [12]:
results = []
twice_expanded = []
#testset_ids = once_expanded.filter(pl.col("volume") < 5).select("volume", "nr_RG").unique()
testset_ids = once_expanded.select("volume", "nr_RG").unique().sort(by="*")

In [ ]:
i = 0

for row in testset_ids.iter_rows(named=True):

    vita_df = once_expanded.filter((pl.col("volume") == row["volume"]) & (pl.col("nr_RG") == row["nr_RG"]))
    once_expanded_text = vita_df_to_text(vita_df)
    candidates = determine_candidates(vita_df, simple, complex)

    if not candidates:
        print(f"skipped vita #{i} (volume {row["volume"]} - nr {row["nr_RG"]}) with no candidates to expand")
        continue

    result = expand_single(once_expanded_text, candidates)
    if result is None:
        print(f"skipped vita #{i} (volume {row["volume"]} - nr {row["nr_RG"]}) with no occurrences of the candidate abbreviations")
        continue

    twice_expanded_text = result["text"]
    twice_expanded_df = text_to_vita_df(twice_expanded_text, row["volume"], row["nr_RG"])

    twice_expanded.append(twice_expanded_df)

    results.append({
        "candidates": candidates,
        "choices": result["choices"],
        "errors": result["errors"],
        "original_text": vita_df_to_text(rg.filter((pl.col("volume") == row["volume"]) & (pl.col("nr_RG") == row["nr_RG"]))),
        "once_expanded_text": once_expanded_text,
        "twice_expanded_text": twice_expanded_text,
    })

    i += 1
    if i % 10 == 0:
        print(f"finished processing {i} vitas")
        #break

twice_expanded = pl.concat(twice_expanded)

In [15]:
len(results), len(twice_expanded)

(156, 539)

In [16]:
with open("data/oss120b/results.json", "w") as file:
    json.dump(results, file, indent=2)

twice_expanded.write_csv("data/oss120b/twice_expanded.csv")

In [57]:

with open("data/gemma4/results.json", "r") as file:
    r = json.load(file)

t = pl.read_csv("data/gemma4/twice_expanded.csv")

In [59]:
pattern = r"[a-zA-Z]{2,200}\."

# filtering for all volumes except 10, because we don't have any rules for volumes 10 yet and are dropping this one when expanding
counts_rg = rg.filter(pl.col("volume") < 10).with_columns(pl.col("header_no_tags").str.count_matches(pattern).sum().alias("countH"), pl.col("regest_no_tags").str.count_matches(pattern).sum().alias("countR"))
abbreviations_rg = counts_rg.row(0)[-2] + counts_rg.row(0)[-1]

counts_once = once_expanded.filter(pl.col("volume") != 10).with_columns(pl.col("header_no_tags").str.count_matches(pattern).sum().alias("countH"), pl.col("regest_no_tags").str.count_matches(pattern).sum().alias("countR"))
abbreviations_once = counts_once.row(0)[-2] + counts_once.row(0)[-1]

counts_twice = twice_expanded.filter(pl.col("volume") != 10).with_columns(pl.col("header_no_tags").str.count_matches(pattern).sum().alias("countH"), pl.col("regest_no_tags").str.count_matches(pattern).sum().alias("countR"))
abbreviations_twice = counts_twice.row(0)[-2] + counts_twice.row(0)[-1]

print(f"abbreviations rg: {abbreviations_rg}")
print(f"abbreviations once: {abbreviations_once}")
print(f"abbreviations twice: {abbreviations_twice}")

abbreviations rg: 4360
abbreviations once: 2189
abbreviations twice: 684


In [67]:
once_expanded.filter(pl.col("regest_no_tags").str.contains("als Ablativus")).select("volume", "nr_RG").unique()
rg.join(with_errors, on=["volume","nr_RG"], how="anti").filter(pl.col("volume") < 10)

volume,nr_RG,nr_suffix,header_no_tags,regest_no_tags,id_RG_all
i64,i64,i64,str,str,str
2,370,0,"""Albertus aep. etc. Magdeburg e…",null,"""10200370-0"""
2,370,1,null,"""conc. indulg. iubilei visitant…","""10200370-1"""
2,370,2,null,"""et Benedictus abb. s. Petri de…","""10200370-2"""
2,370,3,null,"""conserv. 17 dec. 1396 L 45 122…","""10200370-3"""
2,370,4,null,"""m. evoc. can. Magdeburg. qui r…","""10200370-4"""
…,…,…,…,…,…
9,5702,2,null,"""Precept. et fratres dom. s. An…","""10905702-2"""
9,6104,0,"""Wernigerad ac Stalberg""",null,"""10906104-0"""
9,6104,1,null,"""Colleg. eccl. in op. W. Halber…","""10906104-1"""
